In [ ]:
%pip install -q -U git+https://github.com/huggingface/transformers accelerate pillow qwen-vl-utils "datasets==3.6.0"


In [ ]:
from pathlib import Path
import json
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
BASE_DIR = Path("mme_realworld") if Path("mme_realworld/qwen_outputs").exists() else Path(".")
EXAMPLES_PATH = BASE_DIR / "examples/examples.jsonl"
OUTPUT_DIR = BASE_DIR / "qwen_outputs"

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28
KEEP_RATIO = 0.5
MAX_NEW_TOKENS = 8

if not EXAMPLES_PATH.exists():
    raise FileNotFoundError(f"Missing {EXAMPLES_PATH}; run this notebook from the repo root or mme_realworld folder")

with open(EXAMPLES_PATH, "r", encoding="utf-8") as f:
    samples = sorted((json.loads(line) for line in f if line.strip()), key=lambda x: x["example_id"])

print(f"Loaded {len(samples)} MME-RealWorld examples")


In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda", attn_implementation="eager"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)


In [ ]:
def build_question(sample):
    choices = "\n".join(sample["answer_choices"])
    return (
        "Look carefully at the image and answer the multiple-choice question.\n"
        "Choose the best option from A, B, C, D, and E.\n"
        "Respond with only the option letter, without explanation.\n\n"
        f"Question: {sample['question']}\n\n"
        f"Options:\n{choices}\n\n"
        "Answer:"
    )

def prepare_inputs(sample):
    image_path = BASE_DIR / sample["image_file"].replace("\\", "/")
    image = Image.open(image_path).convert("RGB")
    question = build_question(sample)
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    return {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

def load_vision_importance(sample):
    path = OUTPUT_DIR / f"sample_{sample['example_id']:02d}.pt"
    bundle = torch.load(path, map_location="cpu")
    importance = bundle["debiased"].mean(0).float()
    if importance.numel() != bundle["num_image_tokens"]:
        raise ValueError(f"{path} has {importance.numel()} importance scores but {bundle['num_image_tokens']} image tokens")
    return importance, bundle

def vision_embeds(inputs):
    pixel_values = inputs["pixel_values"].type(model.model.visual.get_dtype() if hasattr(model.model.visual, "get_dtype") else inputs["pixel_values"].dtype)
    with torch.no_grad():
        embeds = model.model.visual(pixel_values, grid_thw=inputs["image_grid_thw"])
    return embeds.pooler_output if hasattr(embeds, "pooler_output") else embeds

def build_pruned_inputs(inputs, vision_importance, keep_ratio=KEEP_RATIO):
    input_ids, attention_mask = inputs["input_ids"], inputs["attention_mask"]
    image_embeds = vision_embeds(inputs)
    image_positions = torch.where(input_ids[0] == model.config.image_token_id)[0]
    if image_embeds.shape[0] != image_positions.numel():
        raise ValueError(f"image_embeds={image_embeds.shape[0]} image_positions={image_positions.numel()}")
    if vision_importance.numel() != image_positions.numel():
        raise ValueError(f"importance={vision_importance.numel()} image_positions={image_positions.numel()}")
    k = max(1, round(image_embeds.shape[0] * keep_ratio))
    keep_idx = torch.sort(torch.topk(vision_importance.to(model.device), k).indices).values
    keep_seq = torch.ones_like(input_ids[0], dtype=torch.bool, device=model.device)
    keep_seq[image_positions] = False
    keep_seq[image_positions[keep_idx]] = True
    text_embeds = model.model.language_model.embed_tokens(input_ids)
    new_ids = input_ids[:, keep_seq]
    new_embeds = text_embeds[:, keep_seq].clone()
    new_embeds[0, new_ids[0] == model.config.image_token_id] = image_embeds[keep_idx].to(new_embeds.dtype)
    position_ids, rope_deltas = model.model.get_rope_index(
        input_ids=input_ids,
        image_grid_thw=inputs["image_grid_thw"],
        attention_mask=attention_mask,
        mm_token_type_ids=inputs["mm_token_type_ids"],
    )
    return {
        "input_ids": new_ids,
        "inputs_embeds": new_embeds,
        "attention_mask": attention_mask[:, keep_seq],
        "mm_token_type_ids": inputs["mm_token_type_ids"][:, keep_seq],
        "position_ids": position_ids[:, :, keep_seq],
        "rope_deltas": rope_deltas,
        "keep_idx": keep_idx,
    }

def run_pruned(inputs, vision_importance, keep_ratio=KEEP_RATIO, max_new_tokens=MAX_NEW_TOKENS):
    pruned = build_pruned_inputs(inputs, vision_importance, keep_ratio)
    with torch.no_grad():
        ids = model.generate(
            inputs_embeds=pruned["inputs_embeds"],
            attention_mask=pruned["attention_mask"],
            position_ids=pruned["position_ids"],
            rope_deltas=pruned["rope_deltas"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    answer = processor.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return ids, answer, pruned


In [ ]:
sample_idx = 0
sample = samples[sample_idx]
importance, bundle = load_vision_importance(sample)
inputs = prepare_inputs(sample)
generated_ids, answer, pruned = run_pruned(inputs, importance)
print("sample_id=", sample["example_id"])
print("question=", sample["question"])
print("ground_truth=", sample["ground_truth"])
print("unpruned_answer=", bundle["model_answer"])
print("pruned_answer=", answer)
print("kept_tokens=", pruned["keep_idx"].numel(), "/", importance.numel())
print("pruned_input_shape=", tuple(pruned["input_ids"].shape))
